In [0]:
from pyspark.sql import functions as sf
from pyspark.sql.types import DoubleType

df = spark.table("medical_data.bronze.procedures") \
    .select("start", "stop", "patient", "encounter_id", "code", "description", "base_cost") 

df_clean = df \
    .fillna({
        "base_cost": 0,
        "code": "NA",
        "description": "NA",
        "encounter_id": "NA",
        "patient": "NA"
    }) \
    .withColumn("base_cost", sf.regexp_replace(sf.col("base_cost"), ",", "").cast(DoubleType())) \
    .dropDuplicates(["encounter_id", "code"]) \
    .filter(sf.col("encounter_id").isNotNull() & sf.col("patient").isNotNull())

df_with_features = df_clean \
    .withColumn("year", sf.year(sf.to_timestamp("stop"))) \
    .withColumn("half_yearly", sf.ceil(sf.quarter(sf.to_timestamp("stop"))/2))

df_with_features.write.mode("overwrite").saveAsTable("medical_data.silver.procedures_s")